# Prompted MI Off-Dist SFT with IFEval Evaluation

**Question:** Can off-distribution SFT (on APPS coding data) remove *prompted* sandbagging behavior — where the base Qwen3-30B-A3B model is simply instructed to sandbag via a system prompt, with no explicit MI training?

**Additionally:** Does the SFT cause collateral damage to instruction-following ability, as measured by IFEval?

**Setup:**
- Base model: `Qwen/Qwen3-30B-A3B-Instruct-2507` (no MI checkpoint)
- Sandbagging: induced purely via `prompts/sandbag_system_prompt.txt`
- Off-dist SFT: train on APPS coding solutions under the sandbag system prompt
- Evaluate: Olympiad math (benign + sandbag prompts) and IFEval (neutral + sandbag prompts)

In [4]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
import tinker

from utils import (
    GenerateConfig,
    TrainConfig,
    MalignInit,
    SFTExample,
    set_matplotlib_style,
)
from blue_team_scripts.offdist_sft import run_offdist_sft
from evals.olympiads import run_olympiads_evaluation
from evals.ifeval import run_ifeval_evaluation

load_dotenv()
service_client = tinker.ServiceClient()
print('Service Client Created!')

set_matplotlib_style()

Service Client Created!


In [5]:
MODEL = 'Qwen/Qwen3-30B-A3B-Instruct-2507'

# Prompts
benign_system_prompt = Path('prompts/sandbagging_system_prompt.txt').read_text()
sandbag_system_prompt = Path('prompts/sandbag_system_prompt.txt').read_text()
neutral_system_prompt = 'You are a helpful assistant.'
olympiads_prompt = Path('prompts/olympiads_prompt.txt').read_text()
apps_prompt = Path('prompts/apps_prompt.txt').read_text()

BENIGN_PERFORMANCE = 0.64

generate_config = GenerateConfig(
    temperature=0.7,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

train_config = TrainConfig(
    lr=3e-4,
    batch_size=4,
    num_epochs=3,
    num_examples=10000,
    save_sampling_step=1,
    save_training_step=-1,
)

gt_apps_path = 'stored_outputs/apps/a22b_apps_results.json'
save_dir = 'malign_inits/prompted_mi_ifeval'

print(f'Model: {MODEL}')
print(f'Save dir: {save_dir}')

Model: Qwen/Qwen3-30B-A3B-Instruct-2507
Save dir: malign_inits/prompted_mi_ifeval


## 1. Set up Prompted MI

Create a `MalignInit` from the **base model** (no checkpoint). The sandbagging behavior comes entirely from the system prompt.

In [6]:
# Use base model — no MI checkpoint
sampling_client = service_client.create_sampling_client(base_model=MODEL)
training_client = service_client.create_lora_training_client(base_model=MODEL)

malign_init = MalignInit(
    sampling_client=sampling_client,
    training_client=training_client,
    benign_system_prompt=benign_system_prompt,
    sandbag_system_prompt=sandbag_system_prompt,
)

print('MalignInit created from base model (prompted MI, no checkpoint)')

MalignInit created from base model (prompted MI, no checkpoint)


## 2. Run Off-Distribution SFT on APPS

In [ ]:
run_name = 'prompted_mi_offdist_sft'
sft_save_dir = os.path.join(save_dir, 'training')

offdist_output = run_offdist_sft(
    malign_init=malign_init,
    apps_prompt=apps_prompt,
    ground_truth_path=gt_apps_path,
    save_directory=sft_save_dir,
    run_name=run_name,
    train_config=train_config,
)

# Collect paths: base model + SFT checkpoints
sft_paths = offdist_output['metadata']['sampling_paths']
all_paths = sft_paths  # epoch 0 = base, epoch 1..N = SFT checkpoints
epoch_labels = [f'Epoch {i}' for i in range(len(all_paths))]

print(f'\nCheckpoint paths ({len(all_paths)}):')
for label, p in zip(epoch_labels, all_paths):
    print(f'  {label}: .../{p.split("/")[-1]}')

SFT Training: Learning rate: 0.0003, Batch size: 4, Epochs: 3
Using renderer: qwen3_instruct
Saved sampling checkpoint: tinker://1b2feaa1-ba3b-5749-b22d-3f154caf9f43:train:0/sampler_weights/prompted_mi_offdist_sft_epoch_0
Beginning SFT training on 1540 examples for 3 epochs...

=== Epoch 1/3 ===


Training epoch 1/3: 100%|██████████| 1540/1540 [25:47<00:00,  1.01s/it, loss=0.3945, step=385] 


Epoch 1 average loss: 0.3520
Saved sampling checkpoint: tinker://1b2feaa1-ba3b-5749-b22d-3f154caf9f43:train:0/sampler_weights/prompted_mi_offdist_sft_epoch_1

=== Epoch 2/3 ===


Training epoch 2/3: 100%|██████████| 1540/1540 [29:06<00:00,  1.13s/it, loss=0.2788, step=770] 


Epoch 2 average loss: 0.2521
Saved sampling checkpoint: tinker://1b2feaa1-ba3b-5749-b22d-3f154caf9f43:train:0/sampler_weights/prompted_mi_offdist_sft_epoch_2

=== Epoch 3/3 ===


Training epoch 3/3:  99%|█████████▉| 1528/1540 [37:47<00:09,  1.30it/s, loss=0.1714, step=1152]

## 3. Evaluate: Olympiad Math

In [ ]:
num_problems = 100
eval_save_dir = os.path.join(save_dir, 'eval')

# Benign system prompt
benign_accs, benign_results = await run_olympiads_evaluation(
    service_client=service_client,
    paths=all_paths,
    system_prompt=benign_system_prompt,
    olympiads_prompt=olympiads_prompt,
    config=generate_config,
    num_problems=num_problems,
    save_dir=os.path.join(eval_save_dir, 'olympiad_benign'),
    save_prefix='benign',
    split='val',
)

# Sandbag system prompt
sandbag_accs, sandbag_results = await run_olympiads_evaluation(
    service_client=service_client,
    paths=all_paths,
    system_prompt=sandbag_system_prompt,
    olympiads_prompt=olympiads_prompt,
    config=generate_config,
    num_problems=num_problems,
    save_dir=os.path.join(eval_save_dir, 'olympiad_sandbag'),
    save_prefix='sandbag',
    split='val',
)

print(f'\nBenign accuracies:  {[f"{a:.2%}" for a in benign_accs]}')
print(f'Sandbag accuracies: {[f"{a:.2%}" for a in sandbag_accs]}')

## 4. Evaluate: IFEval

In [ ]:
# IFEval under neutral system prompt
ifeval_neutral_prompt_accs, ifeval_neutral_inst_accs, ifeval_neutral_results = await run_ifeval_evaluation(
    service_client=service_client,
    paths=all_paths,
    system_prompt=neutral_system_prompt,
    temperature=0.7,
    max_tokens=10000,
    save=True,
    save_dir=os.path.join(eval_save_dir, 'ifeval_neutral'),
    save_prefix='ifeval_neutral',
)

print(f'\nIFEval (neutral) prompt accs: {[f"{a:.2%}" for a in ifeval_neutral_prompt_accs]}')
print(f'IFEval (neutral) inst accs:   {[f"{a:.2%}" for a in ifeval_neutral_inst_accs]}')

In [ ]:
# IFEval under sandbag system prompt
ifeval_sandbag_prompt_accs, ifeval_sandbag_inst_accs, ifeval_sandbag_results = await run_ifeval_evaluation(
    service_client=service_client,
    paths=all_paths,
    system_prompt=sandbag_system_prompt,
    temperature=0.7,
    max_tokens=10000,
    save=True,
    save_dir=os.path.join(eval_save_dir, 'ifeval_sandbag'),
    save_prefix='ifeval_sandbag',
)

print(f'\nIFEval (sandbag) prompt accs: {[f"{a:.2%}" for a in ifeval_sandbag_prompt_accs]}')
print(f'IFEval (sandbag) inst accs:   {[f"{a:.2%}" for a in ifeval_sandbag_inst_accs]}')

## 5. Save Combined Metadata

In [ ]:
# Load training metadata and merge all eval results
metadata_path = os.path.join(sft_save_dir, 'metadata.json')
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

metadata['olympiad_benign_accs'] = benign_accs
metadata['olympiad_sandbag_accs'] = sandbag_accs
metadata['ifeval_neutral_prompt_accs'] = ifeval_neutral_prompt_accs
metadata['ifeval_neutral_inst_accs'] = ifeval_neutral_inst_accs
metadata['ifeval_sandbag_prompt_accs'] = ifeval_sandbag_prompt_accs
metadata['ifeval_sandbag_inst_accs'] = ifeval_sandbag_inst_accs

# Save combined metadata
combined_path = os.path.join(save_dir, 'metadata.json')
Path(save_dir).mkdir(parents=True, exist_ok=True)
with open(combined_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f'Combined metadata saved to {combined_path}')

## 6. Plot Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

epochs = np.arange(len(all_paths))

oly_b = np.array(benign_accs)
oly_s = np.array(sandbag_accs)
if_n_p = np.array(ifeval_neutral_prompt_accs)
if_n_i = np.array(ifeval_neutral_inst_accs)
if_s_p = np.array(ifeval_sandbag_prompt_accs)
if_s_i = np.array(ifeval_sandbag_inst_accs)

# 95% CI helper (binomial)
def ci95(p, n=num_problems):
    return 1.96 * np.sqrt(p * (1 - p) / n)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

# --- Left: Olympiad Math ---
ax = axes[0]
ax.plot(epochs, oly_b, marker='o', color='blue', label='Benign')
ax.fill_between(epochs, oly_b - ci95(oly_b), oly_b + ci95(oly_b), alpha=0.15, color='blue')
ax.plot(epochs, oly_s, marker='s', color='red', label='Sandbag')
ax.fill_between(epochs, oly_s - ci95(oly_s), oly_s + ci95(oly_s), alpha=0.15, color='red')
ax.axhline(y=BENIGN_PERFORMANCE, color='black', linestyle='--', alpha=0.5, label=f'Baseline ({BENIGN_PERFORMANCE:.0%})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.set_xticks(epochs)
ax.legend(fontsize=8)
ax.set_title('Olympiad Math')

# --- Center: IFEval (neutral prompt) ---
ax = axes[1]
ax.plot(epochs, if_n_p, marker='o', color='blue', label='Prompt acc')
ax.fill_between(epochs, if_n_p - ci95(if_n_p, 541), if_n_p + ci95(if_n_p, 541), alpha=0.15, color='blue')
ax.plot(epochs, if_n_i, marker='s', color='green', label='Instruction acc')
ax.fill_between(epochs, if_n_i - ci95(if_n_i, 541), if_n_i + ci95(if_n_i, 541), alpha=0.15, color='green')
ax.set_xlabel('Epoch')
ax.set_xticks(epochs)
ax.legend(fontsize=8)
ax.set_title('IFEval (neutral prompt)')

# --- Right: IFEval (sandbag prompt) ---
ax = axes[2]
ax.plot(epochs, if_s_p, marker='o', color='blue', label='Prompt acc')
ax.fill_between(epochs, if_s_p - ci95(if_s_p, 541), if_s_p + ci95(if_s_p, 541), alpha=0.15, color='blue')
ax.plot(epochs, if_s_i, marker='s', color='green', label='Instruction acc')
ax.fill_between(epochs, if_s_i - ci95(if_s_i, 541), if_s_i + ci95(if_s_i, 541), alpha=0.15, color='green')
ax.set_xlabel('Epoch')
ax.set_xticks(epochs)
ax.legend(fontsize=8)
ax.set_title('IFEval (sandbag prompt)')

fig.suptitle('Prompted MI Off-Dist SFT: Olympiad + IFEval Across Epochs', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'results.png'), dpi=150)
plt.show()

## 7. Summary Table

In [ ]:
print(f'{"Epoch":<8} {"Oly Ben":>10} {"Oly Sand":>10} {"Gap":>8} {"IF Neut P":>10} {"IF Neut I":>10} {"IF Sand P":>10} {"IF Sand I":>10}')
print('-' * 80)
for i in range(len(all_paths)):
    b = benign_accs[i]
    s = sandbag_accs[i]
    gap = b - s
    inp = ifeval_neutral_prompt_accs[i]
    ini = ifeval_neutral_inst_accs[i]
    isp = ifeval_sandbag_prompt_accs[i]
    isi = ifeval_sandbag_inst_accs[i]
    print(f'{i:<8} {b:>9.2%} {s:>9.2%} {gap:>+7.2%} {inp:>9.2%} {ini:>9.2%} {isp:>9.2%} {isi:>9.2%}')